# Exploratory Data Analysis: Jane Street Market Data

This notebook walks through a systematic exploration of the Jane Street Real-Time Market Data Forecasting dataset. The goal is to understand the data's structure, identify patterns, and inform modeling decisions.

**Key questions:**
1. What does the target distribution look like, and how do sample weights affect it?
2. How much missing data exists, and which features are affected?
3. Are there temporal patterns or regime changes in the data?
4. Do different symbols (financial instruments) behave differently?
5. Which features are most correlated with the target?

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

from config import Config
from src.visualization import apply_style, plot_target_distribution, plot_feature_distributions
from src.visualization import plot_missing_data, plot_correlation_heatmap, plot_target_correlations
from src.visualization import plot_temporal_patterns, plot_symbol_analysis
from src.feature_analysis import (
    missing_data_profile, compute_correlations, find_highly_correlated,
    compute_summary_statistics, weight_analysis, per_symbol_summary
)

apply_style()
cfg = Config()

%matplotlib inline

## 1. Load and Inspect the Data

In [ ]:
path = cfg.data.partition_path(0)
df_pl = pl.read_parquet(str(path))
df = df_pl.to_pandas()

print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"\nColumn types:\n{df.dtypes.value_counts()}")
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
df.head()

In [ ]:
feature_cols = [f'feature_{i:02d}' for i in range(79)]
target_col = 'responder_6'

summary = compute_summary_statistics(df)
summary.head(20)

## 2. Missing Data Analysis

Understanding missing patterns is critical before choosing an imputation strategy. Zero-fill (our current approach) may not be optimal if missingness is informative.

In [ ]:
profile = missing_data_profile(df)
missing_features = profile[profile['missing_pct'] > 0]

print(f"Features with missing values: {len(missing_features)} / {len(feature_cols)}")
print(f"Max missing rate: {profile['missing_pct'].max():.1f}%")

plot_missing_data(profile['missing_pct'])
plt.show()

## 3. Target Variable Analysis

The target (`responder_6`) represents a financial return. Key things to check:
- Is it centered near zero? (expected for returns)
- Are there fat tails? (common in financial data)
- How do the sample weights shift the effective distribution?

In [ ]:
targets = df[target_col].fillna(0).values
weights = df['weight'].fillna(0).values

print(f"Target stats:")
print(f"  Mean:     {targets.mean():.6f}")
print(f"  Median:   {np.median(targets):.6f}")
print(f"  Std:      {targets.std():.6f}")
print(f"  Skew:     {pd.Series(targets).skew():.4f}")
print(f"  Kurtosis: {pd.Series(targets).kurtosis():.4f}")
print(f"\n  Weighted mean: {np.average(targets, weights=weights):.6f}")

plot_target_distribution(targets, weights)
plt.show()

## 4. Feature Distributions

A quick visual scan of feature distributions helps spot anomalies: binary/discrete features, extreme outliers, or features with very little variance.

In [ ]:
plot_feature_distributions(df, feature_cols[:20], ncols=5)
plt.show()

In [ ]:
plot_feature_distributions(df, feature_cols[20:40], ncols=5)
plt.show()

## 5. Correlation Analysis

High inter-feature correlation can cause multicollinearity issues for linear models and indicates redundancy that tree-based models handle naturally.

Feature-target correlations tell us which raw features have the most linear predictive power.

In [ ]:
corr_matrix, target_corr = compute_correlations(df, feature_cols, target_col)

plot_correlation_heatmap(corr_matrix)
plt.show()

In [ ]:
plot_target_correlations(target_corr, top_n=25)
plt.show()

print("\nTop 10 positive correlations with target:")
print(target_corr.head(10).to_string())
print("\nTop 10 negative correlations with target:")
print(target_corr.tail(10).to_string())

In [ ]:
highly_corr = find_highly_correlated(corr_matrix, threshold=0.90)
print(f"Feature pairs with |r| >= 0.90: {len(highly_corr)}")
if len(highly_corr) > 0:
    print(highly_corr.head(15).to_string(index=False))

## 6. Temporal Patterns

Financial markets have non-stationary dynamics. Checking for trend, volatility clustering, and regime changes helps decide whether online learning is necessary.

In [ ]:
plot_temporal_patterns(df)
plt.show()

## 7. Per-Symbol Analysis

Each `symbol_id` represents a different financial instrument. If symbols have very different statistical properties, per-symbol models (mixture of experts) or symbol embeddings should help.

In [ ]:
plot_symbol_analysis(df)
plt.show()

In [ ]:
sym_summary = per_symbol_summary(df)
print(f"Number of symbols: {len(sym_summary)}")
print(f"\nSample count range: {sym_summary['count'].min():,} - {sym_summary['count'].max():,}")
print(f"Target mean range:  {sym_summary['target_mean'].min():.6f} - {sym_summary['target_mean'].max():.6f}")
print(f"Target std range:   {sym_summary['target_std'].min():.6f} - {sym_summary['target_std'].max():.6f}")
sym_summary

## 8. Weight Distribution

Sample weights directly affect the evaluation metric (weighted R²). Understanding their distribution helps interpret model performance.

In [ ]:
w_stats = weight_analysis(df)
for k, v in w_stats.items():
    print(f"  {k:>15s}: {v:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['weight'].dropna(), bins=100, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Weight')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Weight Distribution')

sym_weights = df.groupby('symbol_id')['weight'].mean()
axes[1].bar(sym_weights.index, sym_weights.values, edgecolor='black')
axes[1].set_xlabel('Symbol ID')
axes[1].set_ylabel('Mean Weight')
axes[1].set_title('Mean Weight by Symbol')

plt.tight_layout()
plt.show()

## 9. Key Takeaways

Summarize findings that will inform modeling decisions:

| Finding | Implication |
|---|---|
| Target is near-zero mean with fat tails | Financial returns — regression task, not classification. Huber loss may help with outliers. |
| Missing data in several features | Zero-fill is simple but may lose signal. Missingness itself could be a feature. |
| Temporal volatility clustering | Distribution shift over time — online learning should outperform static models. |
| Symbols have different target statistics | Per-symbol experts or learned symbol embeddings can capture this heterogeneity. |
| Several highly correlated feature pairs | Feature redundancy exists — tree models (XGBoost) handle this natively; for NNs, PCA or dropout may help. |
| Weights are non-uniform | High-weight predictions matter more — weighted loss functions are essential. |